# AIDP capability probe — native masking & zero-copy clone

**What this proves.** A conservative, *runnable* check of what Oracle AI Data Platform's Spark 3.5.0 / Delta engine
supports **natively** for two governance/data-management capabilities that customers migrating from **Snowflake** or
**Databricks** ask about:

1. **Tag/policy-based column masking** — declarative masking bound to a column tag or policy
   (Snowflake *masking policies* + tag-based masking; Databricks Unity Catalog *column masks*).
2. **Zero-copy clone** — Snowflake `CREATE TABLE ... CLONE`; the lakehouse analogue is Delta `SHALLOW`/`DEEP CLONE`.
3. **Tagging in SQL** — *attaching* a tag (works) vs *enforcing* it (you build the bridge). Part C tests this and
   demonstrates a tag-registry → generated-masking-view **bridge** as the alternative to native tag-based masking.

Each test uses a safe `run()` wrapper so an *unsupported* statement is recorded as `UNSUPPORTED/ERROR` instead of
aborting the notebook. Read the final summary cell for the verdict.

> Verified live on `navid-dev-sandbox` (us-ashburn-1) — see `README.md` for the recorded control-plane result.

## Part A — Control-plane API probe (run from a workstation, not the kernel)

There is **no data-plane masking/classification REST API** in the tested tenancy: the endpoints you'd expect for
tag-based masking all return **404** while `/roles` and `/catalogs` return **200** (so it's feature-absence, not auth).
Reproduce with the OCI CLI from your laptop (needs a valid api-key profile):

```bash
OCID=<your DataLake OCID>
BASE="https://aidp.us-ashburn-1.oci.oraclecloud.com/20240831/dataLakes/$OCID"
for ep in roles catalogs maskingPolicies columnMaskingPolicies dataClassifications \
          tags classifications maskingRules policies dataProtection sensitiveDataTypes; do
  s=$(oci raw-request --http-method GET --target-uri "$BASE/$ep" --profile DEFAULT 2>&1 \
        | grep '"status"' | sed 's/.*"status": *"\([^"]*\)".*/\1/')
  printf '%-22s -> %s\n' "$ep" "$s"
done
# roles/catalogs -> 200 OK ; every masking/tag/classification endpoint -> 404 Not Found
```

## Part B — Data-plane (SQL) capability tests

Run the cells below on an AIDP cluster.

In [ ]:
# Config + safe-run harness
CATALOG = 'default'          # adjust if your catalog differs
DB = 'mask_clone_probe'      # scratch schema this notebook creates & drops

results = []
def run(label, sql, show=False):
    try:
        df = spark.sql(sql)
        rows = df.collect() if show else None
        out = [r.asDict() for r in rows] if rows is not None else 'OK'
        results.append((label, 'SUPPORTED/OK', out))
        print(f'[SUPPORTED] {label}')
        if isinstance(out, list):
            for r in out: print('    ', r)
    except Exception as e:
        msg = str(e).splitlines()[0][:300]
        results.append((label, 'UNSUPPORTED/ERROR', msg))
        print(f'[UNSUPPORTED] {label}\n     -> {msg}')

In [ ]:
# Setup: a tiny table with 'PII' columns
run('setup: drop schema',  f'DROP SCHEMA IF EXISTS {CATALOG}.{DB} CASCADE')
run('setup: create schema',f'CREATE SCHEMA IF NOT EXISTS {CATALOG}.{DB}')
run('setup: create table', f'''CREATE TABLE {CATALOG}.{DB}.customers
    (id INT, name STRING, email STRING, ssn STRING) USING delta''')
run('setup: insert', f'''INSERT INTO {CATALOG}.{DB}.customers VALUES
    (1,'Alice','alice@x.com','111-22-3333'),
    (2,'Bob','bob@x.com','444-55-6666')''')

### Test 1 — Unity-Catalog-style **column mask** (`ALTER COLUMN ... SET MASK`)
Expected to be **unsupported** on open-source Delta/Spark (this is a Databricks-managed feature).

In [ ]:
run('T1a: CREATE FUNCTION mask udf',
    f"CREATE OR REPLACE FUNCTION {CATALOG}.{DB}.mask_ssn(s STRING) RETURNS STRING RETURN 'XXX-XX-' || right(s,4)")
run('T1b: ALTER COLUMN SET MASK (UC syntax)',
    f'ALTER TABLE {CATALOG}.{DB}.customers ALTER COLUMN ssn SET MASK {CATALOG}.{DB}.mask_ssn')

### Test 2 — Tag-based / policy masking DDL
Snowflake `CREATE MASKING POLICY`, column `SET TAGS`, and `SET ROW FILTER`. Expected **unsupported**.

In [ ]:
run('T2a: SET TAGS on column',
    f"ALTER TABLE {CATALOG}.{DB}.customers ALTER COLUMN ssn SET TAGS ('pii'='true')")
run('T2b: CREATE MASKING POLICY (Snowflake-style)',
    f"CREATE MASKING POLICY {DB}_ssn_pol AS (v STRING) RETURNS STRING -> 'XXX'")
run('T2c: SET ROW FILTER',
    f'ALTER TABLE {CATALOG}.{DB}.customers SET ROW FILTER {CATALOG}.{DB}.mask_ssn ON (ssn)')

### Test 3 — Restricted / redacting **VIEW** (the supported workaround)
The practical column-level control today: expose a view that redacts, grant on the view, not the base table.

In [ ]:
run('T3a: create redacting view',
    f'''CREATE OR REPLACE VIEW {CATALOG}.{DB}.customers_masked AS
        SELECT id, name,
               concat(substr(email,1,1),'***@***') AS email,
               concat('XXX-XX-', right(ssn,4))   AS ssn
        FROM {CATALOG}.{DB}.customers''')
run('T3b: query redacting view',  f'SELECT * FROM {CATALOG}.{DB}.customers_masked ORDER BY id', show=True)
run('T3c: query base (contrast)', f'SELECT * FROM {CATALOG}.{DB}.customers ORDER BY id', show=True)

### Test 4 — **Zero-copy clone** (Delta `SHALLOW CLONE` / `DEEP CLONE`)
`SHALLOW CLONE` = metadata-only, references the source's data files (true zero-copy, copy-on-write).
`DEEP CLONE` = full independent copy (not zero-copy).

> **Operational gotcha vs Snowflake:** a shallow clone references the *source's* parquet files. Running `VACUUM`
> on the source can delete those files and break the clone — Snowflake manages this for you; Delta does not.

In [ ]:
run('T4a: SHALLOW CLONE (zero-copy)',
    f'CREATE OR REPLACE TABLE {CATALOG}.{DB}.customers_shallow SHALLOW CLONE {CATALOG}.{DB}.customers')
run('T4b: shallow clone reads same rows', f'SELECT count(*) AS n FROM {CATALOG}.{DB}.customers_shallow', show=True)
run('T4c: DESCRIBE DETAIL shallow clone',
    f'SELECT format, numFiles, sizeInBytes, location FROM (DESCRIBE DETAIL {CATALOG}.{DB}.customers_shallow)', show=True)
run('T4d: history shows CLONE op',
    f'SELECT operation FROM (DESCRIBE HISTORY {CATALOG}.{DB}.customers_shallow) ORDER BY version LIMIT 3', show=True)
run('T4e: insert into clone (COW divergence)',
    f"INSERT INTO {CATALOG}.{DB}.customers_shallow VALUES (99,'Zoe','zoe@x.com','999-88-7777')")
run('T4f: clone count after insert (expect 3)', f'SELECT count(*) AS n FROM {CATALOG}.{DB}.customers_shallow', show=True)
run('T4g: SOURCE count unchanged (expect 2)',   f'SELECT count(*) AS n FROM {CATALOG}.{DB}.customers', show=True)
run('T4h: DEEP CLONE (full copy)',
    f'CREATE OR REPLACE TABLE {CATALOG}.{DB}.customers_deep DEEP CLONE {CATALOG}.{DB}.customers')
run('T4i: deep clone reads same rows', f'SELECT count(*) AS n FROM {CATALOG}.{DB}.customers_deep', show=True)

### Test 5 — Clone **structure without data** (Snowflake `CREATE TABLE ... LIKE` / empty clone)

In [ ]:
run('T5a: CREATE TABLE ... LIKE (no data)',
    f'CREATE TABLE {CATALOG}.{DB}.customers_empty LIKE {CATALOG}.{DB}.customers')
run('T5b: LIKE clone count (expect 0)',  f'SELECT count(*) AS n FROM {CATALOG}.{DB}.customers_empty', show=True)
run('T5c: LIKE clone keeps columns',     f'SELECT count(*) AS ncols FROM (DESCRIBE {CATALOG}.{DB}.customers_empty)', show=True)
run('T5d: CTAS empty (WHERE 1=0) alt',
    f'CREATE TABLE {CATALOG}.{DB}.customers_empty2 AS SELECT * FROM {CATALOG}.{DB}.customers WHERE 1=0')
run('T5e: CTAS-empty count (expect 0)',  f'SELECT count(*) AS n FROM {CATALOG}.{DB}.customers_empty2', show=True)

## Part C — Tagging in SQL: **attach** vs. **enforce** (the separation)

"Have a tag in SQL" is really two separate things on AIDP, and only the first is free:

| | What it means | AIDP in SQL |
|---|---|---|
| **Attach** a tag | store a key/value on a table or column | ✅ `TBLPROPERTIES` (table) / column comment / a registry table |
| **Enforce** a tag | tag automatically masks or restricts at query time | ❌ no policy engine — **you build the bridge** |

Snowflake's tag-based masking couples the two (tag → masking policy → automatic redaction). AIDP decouples
them: you can *attach* tags in SQL today, but *enforcement* is something you assemble. Tests T6–T8 attach
tags; T9 is the **bridge** that turns tags into masking.

### T6 — Table-level tags via `TBLPROPERTIES` (attach, table scope)
Standard Delta; a real key/value tag store on the table. Table-scoped and descriptive only.

In [ ]:
run('T6a: SET TBLPROPERTIES (tag the table)',
    f"ALTER TABLE {CATALOG}.{DB}.customers SET TBLPROPERTIES "
    f"('classification'='confidential','owner'='cdo','pii'='true')")
run('T6b: read tags back', f'SHOW TBLPROPERTIES {CATALOG}.{DB}.customers', show=True)

### T7 — Column-level tag via COMMENT (attach, column scope — limited)
Delta/Spark has **no** `ALTER COLUMN ... SET TAGS`; the only per-column SQL slot is the comment.
Workable for one dimension, but you're overloading a free-text field.

In [ ]:
run('T7a: encode a tag in the column comment',
    f"ALTER TABLE {CATALOG}.{DB}.customers ALTER COLUMN ssn COMMENT 'tag:pii;tag:masked'")
run('T7b: read column comment', f'DESCRIBE {CATALOG}.{DB}.customers', show=True)

### T8 — Tag **registry table** (attach, column scope — the pattern that scales)
Make the tags *data you control in SQL*: one governance table, tag columns with plain `INSERT`s,
and query "what's tagged PII across the lakehouse?".

In [ ]:
run('T8a: create tag registry',
    f'''CREATE TABLE IF NOT EXISTS {CATALOG}.{DB}.column_tags (
        catalog_name STRING, schema_name STRING, table_name STRING,
        column_name STRING, tag_key STRING, tag_value STRING) USING delta''')
run('T8b: tag columns via INSERT',
    f"INSERT INTO {CATALOG}.{DB}.column_tags VALUES"
    f" ('{CATALOG}','{DB}','customers','ssn','sensitivity','pii'),"
    f" ('{CATALOG}','{DB}','customers','email','sensitivity','pii')")
run('T8c: discover tagged (PII) columns',
    f"SELECT schema_name, table_name, column_name FROM {CATALOG}.{DB}.column_tags "
    f"WHERE tag_key='sensitivity' AND tag_value='pii' ORDER BY column_name", show=True)

### T9 — **The bridge**: turn tags into masking (alternative to native tag-based masking)
AIDP has no engine to interpret the tags, so we generate the redacting view *from* the registry.
Tag a column `pii` → regenerate → the mask follows. Grant on the generated view, not the base table.
This is the **alternative solution** that reproduces Snowflake tag-based masking behaviour on AIDP.

In [ ]:
# read the tags, build a SELECT that redacts every pii-tagged column, (re)create the view
pii = {r.column_name for r in spark.sql(
    f"SELECT column_name FROM {CATALOG}.{DB}.column_tags "
    f"WHERE table_name='customers' AND tag_value='pii'").collect()}
cols = spark.table(f'{CATALOG}.{DB}.customers').columns
proj = [f"concat('XXX-XX-', right({c},4)) AS {c}" if c in pii else c for c in cols]
ddl = f"CREATE OR REPLACE VIEW {CATALOG}.{DB}.customers_tagmasked AS SELECT {', '.join(proj)} FROM {CATALOG}.{DB}.customers"
print('tagged pii columns :', pii)
print('generated view DDL :', ddl)
run('T9a: generate masking view from tags', ddl)
run('T9b: query tag-generated masked view',
    f'SELECT * FROM {CATALOG}.{DB}.customers_tagmasked ORDER BY id', show=True)

### Summary + cleanup

In [ ]:
print('\n================ CAPABILITY SUMMARY ================\n')
for label, status, _ in results:
    print(f'[{status:18}] {label}')
print('\n===================================================')

In [ ]:
# Drop everything this notebook created
run('cleanup: drop schema', f'DROP SCHEMA IF EXISTS {CATALOG}.{DB} CASCADE')